# 01 Inspect WLASL300 Dataset

## Purpose

This notebook prepares the WLASL300 dataset for model development. It reads the WLASL300 metadata file, checks which videos actually exist in the shared `videos` folder, maps each video to its ASL gloss, and creates the usable video index for the next preprocessing step.

## Why this notebook matters

WLASL metadata can include video IDs that are missing from the downloaded `videos` folder. Training with missing files will break later notebooks, so this notebook creates a clean index containing only videos that are actually available locally.

## Expected input files

This notebook expects your project structure to contain:

```text
data/raw/ASL/WLASL300/nslt_300.json
data/raw/ASL/WLASL300/wlasl_class_list.txt
data/raw/ASL/videos/
```

The `videos` folder is shared across WLASL100, WLASL300, WLASL1000, and WLASL2000 to avoid duplicating large video files.

In [1]:
from pathlib import Path
import json
import pandas as pd

## 1. Set project paths

These paths follow the separated WLASL structure. WLASL300 has its own raw metadata folder, processed folder, label map folder, model folder, and report folder.

In [2]:
PROJECT_ROOT = Path("E:/Be_My_Ear")

DATASET_NAME = "WLASL300"
PREFIX = "wlasl300"
NUM_CLASSES = 300

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "ASL" / DATASET_NAME
VIDEOS_DIR = PROJECT_ROOT / "data" / "raw" / "ASL" / "videos"

META_FILE = RAW_DIR / "nslt_300.json"
CLASS_LIST_FILE = RAW_DIR / "wlasl_class_list.txt"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / DATASET_NAME
LABEL_MAP_DIR = PROJECT_ROOT / "data" / "label_maps" / DATASET_NAME

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
LABEL_MAP_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_INDEX_FILE = PROCESSED_DIR / f"{PREFIX}_video_index.csv"
LABEL_MAP_FILE = LABEL_MAP_DIR / f"asl_{PREFIX}_labels.json"

print("Dataset:", DATASET_NAME)
print("Metadata file exists:", META_FILE.exists(), META_FILE)
print("Class list file exists:", CLASS_LIST_FILE.exists(), CLASS_LIST_FILE)
print("Shared videos folder exists:", VIDEOS_DIR.exists(), VIDEOS_DIR)
print("Video index will save to:", VIDEO_INDEX_FILE)
print("Label map will save to:", LABEL_MAP_FILE)

Dataset: WLASL300
Metadata file exists: True E:\Be_My_Ear\data\raw\ASL\WLASL300\nslt_300.json
Class list file exists: True E:\Be_My_Ear\data\raw\ASL\WLASL300\wlasl_class_list.txt
Shared videos folder exists: True E:\Be_My_Ear\data\raw\ASL\videos
Video index will save to: E:\Be_My_Ear\data\processed\ASL\WLASL300\wlasl300_video_index.csv
Label map will save to: E:\Be_My_Ear\data\label_maps\WLASL300\asl_wlasl300_labels.json


## 2. Load WLASL class names

`wlasl_class_list.txt` maps numeric class IDs to human-readable ASL gloss names. The model will learn numeric label IDs, but we need gloss names for interpretation.

In [3]:
class_names = {}

with open(CLASS_LIST_FILE, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            class_id = int(parts[0])
            gloss = parts[1]
            class_names[class_id] = gloss

print("Total class names loaded:", len(class_names))
list(class_names.items())[:10]

Total class names loaded: 2000


[(0, 'book'),
 (1, 'drink'),
 (2, 'computer'),
 (3, 'before'),
 (4, 'chair'),
 (5, 'go'),
 (6, 'clothes'),
 (7, 'who'),
 (8, 'candy'),
 (9, 'cousin')]

## 3. Load WLASL300 metadata

`nslt_300.json` contains the WLASL300 subset information. Each item includes a video ID and action/class information.

In [4]:
with open(META_FILE, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("Total metadata entries:", len(metadata))

first_video_id = list(metadata.keys())[0]
print("Example video ID:", first_video_id)
print("Example metadata item:", metadata[first_video_id])

Total metadata entries: 5118
Example video ID: 05237
Example metadata item: {'subset': 'train', 'action': [77, 1, 55]}


## 4. Build a video availability table

This creates one row per video in the metadata and checks whether the actual `.mp4` file exists locally.

In [5]:
records = []

for video_id, item in metadata.items():
    original_class_id = item["action"][0]
    gloss = class_names.get(original_class_id, "unknown")

    video_path = VIDEOS_DIR / f"{video_id}.mp4"
    exists = video_path.exists()

    records.append({
        "video_id": video_id,
        "original_class_id": original_class_id,
        "gloss": gloss,
        "video_path": str(video_path),
        "exists": exists
    })

df = pd.DataFrame(records)

print("Total videos in metadata:", len(df))
print("Existing videos:", df["exists"].sum())
print("Missing videos:", (~df["exists"]).sum())
print("Classes in metadata:", df["original_class_id"].nunique())

df.head()

Total videos in metadata: 5118
Existing videos: 2660
Missing videos: 2458
Classes in metadata: 300


,video_id,original_class_id,gloss,video_path,exists
0,05237,77,basketball,E:\Be_My_Ear\data\raw\ASL\videos\05237.mp4,False
1,65096,182,arrive,E:\Be_My_Ear\data\raw\ASL\videos\65096.mp4,True
2,65639,279,environment,E:\Be_My_Ear\data\raw\ASL\videos\65639.mp4,True
3,10112,191,chat,E:\Be_My_Ear\data\raw\ASL\videos\10112.mp4,True
4,55348,247,student,E:\Be_My_Ear\data\raw\ASL\videos\55348.mp4,True


## 5. Keep only usable videos and create label IDs

The model needs label IDs from `0` to `num_classes - 1`. We create new label IDs based on the available WLASL300 glosses.

In [6]:
df_usable = df[df["exists"] == True].copy()

glosses = sorted(df_usable["gloss"].unique())
gloss_to_label_id = {gloss: idx for idx, gloss in enumerate(glosses)}

df_usable["label_id"] = df_usable["gloss"].map(gloss_to_label_id)

print("Usable videos:", len(df_usable))
print("Usable classes:", df_usable["label_id"].nunique())
print("Expected classes:", NUM_CLASSES)

df_usable.head()

Usable videos: 2660
Usable classes: 300
Expected classes: 300


,video_id,original_class_id,gloss,video_path,exists,label_id
1,65096,182,arrive,E:\Be_My_Ear\data\raw\ASL\videos\65096.mp4,True,10
2,65639,279,environment,E:\Be_My_Ear\data\raw\ASL\videos\65639.mp4,True,107
3,10112,191,chat,E:\Be_My_Ear\data\raw\ASL\videos\10112.mp4,True,53
4,55348,247,student,E:\Be_My_Ear\data\raw\ASL\videos\55348.mp4,True,253
5,65092,258,argue,E:\Be_My_Ear\data\raw\ASL\videos\65092.mp4,True,9


## 6. Save WLASL300 video index and label map

These two files are used by the later WLASL300 notebooks.

Output files:

```text
data/processed/ASL/WLASL300/wlasl300_video_index.csv
data/label_maps/WLASL300/asl_wlasl300_labels.json
```

In [7]:
df_usable.to_csv(VIDEO_INDEX_FILE, index=False)

label_map = {}

for _, row in df_usable.drop_duplicates("label_id").iterrows():
    label_id = int(row["label_id"])
    label_map[label_id] = {
        "language": "ASL",
        "dataset": DATASET_NAME,
        "gloss": row["gloss"],
        "display_text": row["gloss"],
        "original_class_id": int(row["original_class_id"])
    }

label_map = dict(sorted(label_map.items(), key=lambda x: x[0]))

with open(LABEL_MAP_FILE, "w", encoding="utf-8") as f:
    json.dump(label_map, f, indent=4)

print("Saved video index to:", VIDEO_INDEX_FILE)
print("Saved label map to:", LABEL_MAP_FILE)
print("Final usable videos:", len(df_usable))
print("Final usable classes:", df_usable["label_id"].nunique())

Saved video index to: E:\Be_My_Ear\data\processed\ASL\WLASL300\wlasl300_video_index.csv
Saved label map to: E:\Be_My_Ear\data\label_maps\WLASL300\asl_wlasl300_labels.json
Final usable videos: 2660
Final usable classes: 300


## 7. Review class distribution

This shows how many usable videos each gloss has. WLASL is usually imbalanced, so this information helps explain model performance later.

In [8]:
class_counts = df_usable["gloss"].value_counts()

print("Class distribution summary")
print("--------------------------")
print("Number of classes:", len(class_counts))
print("Minimum videos per class:", class_counts.min())
print("Maximum videos per class:", class_counts.max())
print("Average videos per class:", round(class_counts.mean(), 2))

print("\nClasses with fewer than 5 usable videos:")
print(class_counts[class_counts < 5])

class_counts.head(30)

Class distribution summary
--------------------------
Number of classes: 300
Minimum videos per class: 5
Maximum videos per class: 16
Average videos per class: 8.87

Classes with fewer than 5 usable videos:
Series([], Name: count, dtype: int64)


gloss
before          16
cool            16
thin            16
go              15
drink           15
help            14
cousin          14
computer        14
who             14
candy           13
trade           13
bed             13
accident        13
thanksgiving    13
tall            13
bowling         13
short           13
last            12
pizza           12
change          12
basketball      12
cold            12
dark            12
what            12
shirt           12
corn            12
call            12
yes             12
later           12
man             12
Name: count, dtype: int64

## Final summary

After this notebook, continue with:

```text
02_extract_wlasl300_keypoints.ipynb
```

That notebook will convert each usable WLASL300 video into MediaPipe keypoint files.